# 04 - LGD, EAD and expected loss

Definitions used throughout:

* **EAD** (exposure at default) = outstanding principal when the loan charges
  off, `funded_amnt - total_rec_prncp`.
* **Recovery** = post charge-off collections, `recoveries`.
* **LGD** = `1 - recoveries / EAD`, clipped to `[0, 1]`.

LGD is only observed on defaulted loans - the classic sample-selection problem.
We model it with a Gamma GLM (log link, the Beta/Gamma-type LGD model) and a
gradient-boosting challenger, then compute `EL = PD x LGD x EAD`.

In [ ]:
import pathlib, sys
ROOT = pathlib.Path.cwd()
while not (ROOT / 'config' / 'config.yaml').exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
import numpy as np
import pandas as pd
from IPython.display import Image, display
pd.set_option('display.width', 180)

In [ ]:
from src.config import Config
from src.pipeline import load_modeling_frame, train_oot_split
from src import features, lgd_ead

cfg = Config.load()
df = load_modeling_frame(cfg)
d = lgd_ead.compute_lgd_ead(df)
display(lgd_ead.lgd_eda_table(d))

In [ ]:
split_year = int(cfg.modeling.get('oot_split_year', 2017))
num, cat = list(features.NUMERIC_FEATURES), list(features.CATEGORICAL_FEATURES)
defaults = d[d['lgd'].notna()]
train_def = defaults[defaults['issue_year'] < split_year]
oot_def = defaults[defaults['issue_year'] >= split_year]
X_tr, y_tr = lgd_ead.prepare_lgd_matrix(train_def, num, cat)
X_te, y_te = lgd_ead.prepare_lgd_matrix(oot_def, num, cat)
print('defaulted loans  train:', len(y_tr), ' oot:', len(y_te))
print('mean realised LGD  train:', round(float(y_tr.mean()), 4), ' oot:', round(float(y_te.mean()), 4))

In [ ]:
gamma_res, gamma_coef = lgd_ead.fit_lgd_gamma(X_tr, y_tr, num, cat)
pred_gamma = lgd_ead.predict_lgd_gamma(gamma_res, X_te, num, cat)
tree = lgd_ead.fit_lgd_tree(X_tr, y_tr, num, cat, seed=cfg.seed)
pred_tree = np.clip(np.asarray(tree.predict(X_te), dtype=float), 0.0, 1.0)
comparison = pd.DataFrame([
    {'model': 'gamma_glm', **lgd_ead.regression_metrics(y_te.to_numpy(), pred_gamma)},
    {'model': 'lightgbm', **lgd_ead.regression_metrics(y_te.to_numpy(), pred_tree)},
])
display(comparison.round(4))
display(gamma_coef.head(12).round(4))

Both models struggle to explain realised LGD from origination characteristics
alone (`R^2` is low). That is the expected finding: recovery outcomes are driven
by post-default collection behavior, not by who the borrower was at
origination. The Gamma GLM is retained for the portfolio calculation because it
is monotone and interpretable.

In [ ]:
for name in ['lgd_distribution.png', 'lgd_by_grade.png', 'lgd_model_comparison.png', 'el_by_grade.png']:
    p = ROOT / 'reports' / 'figures' / name
    if p.exists():
        display(Image(filename=str(p)))

In [ ]:
RES = ROOT / 'reports' / 'results'
if (RES / 'el_by_grade.csv').exists():
    display(pd.read_csv(RES / 'el_by_grade.csv').round(4))
    display(pd.read_csv(RES / 'el_portfolio.csv').round(4))

### Notes

* EAD for the expected-loss aggregation uses the **funded amount** as an
  exposure proxy for the whole out-of-time book. The realised `ead` column
  (outstanding principal at charge-off) is used for the LGD target itself.
* LGD predictions are clipped to `[0, 1]`; a Beta regression on a squeezed
  response is the natural extension and is left as future work.